### ReAct Agent

In [ ]:
import openai
import re
import httpx
import os
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI
_ = load_dotenv(find_dotenv())

#### Sending a simple request to the LLM model

In [72]:
# Groq is compatible with how we initialize and use the client for OpenAI
client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1"
)

In [73]:
# request sent to Groq servers, and then Groq serers send it to LLM (llama)
# model receives something like
#     User: Hello world
#     Assistant:
chat_completion = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": "Hello world"}]
)

In [74]:
chat_completion.choices[0].message.content

'Hello world. Is there something I can assist you with?'

### Creating an Agent



In [75]:
#__call__ used to provide the agent with user input. It append user message to conversation history, call the LLM with full history of messages so far, append response from the LLM
# and append it to message history

class Agent:
    def __init__(self, system=""):
        self.system = system
        self.messages = []
        if self.system:
            self.messages.append({"role": "system", "content": self.system})


    def __call__(self, message):
        self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        chat_completion = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            temperature=0,
            messages=self.messages
        )
        return chat_completion.choices[0].message.content

In [76]:
# this is the prompt we give to LLM on how we want the LLM to behave.
# We instruct LLM about

# behavior
# rules
# available tools
# response style

# We keep this in agent message history and the key will be system
# sometimes the LLM may not PAUSE and it calculates the answer itself. You have to write the prompt so that it behaves accordingly




prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE. You have to PAUSE and avoid making up things.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

average_dog_weight:
e.g. average_dog_weight: Collie
returns average weight of a dog when given the breed

Example session:

Question: How much does a Bulldog weigh?
Thought: I should look the dogs weight using average_dog_weight
Action: average_dog_weight: Bulldog
PAUSE

You will be called again with this:

Observation: A Bulldog weights 51 lbs

You then output:

Answer: A bulldog weights 51 lbs
""".strip()

In [77]:
#These are the tools which we provide to LLM

def calculate(what):
    return eval(what)

def average_dog_weight(name):
    if name in "Scottish Terrier":
        return("Scottish Terriers average 20 lbs")
    elif name in "Border Collie":
        return("a Border Collies average weight is 37 lbs")
    elif name in "Toy Poodle":
        return("a toy poodles average weight is 7 lbs")
    else:
        return("An average dog weights 50 lbs")

known_actions = {
    "calculate": calculate,
    "average_dog_weight": average_dog_weight
}

In [78]:
zarvis = Agent(prompt)

In [79]:
result = zarvis("How much does a toy poodle weigh?")
print(result)

Thought: I should look the toy poodle's weight using average_dog_weight
Action: average_dog_weight: Toy Poodle
PAUSE


In [80]:
result = average_dog_weight("Toy Poodle")

In [81]:
result

'a toy poodles average weight is 7 lbs'

In [82]:
next_prompt = "Observation: {}".format(result)
next_prompt

'Observation: a toy poodles average weight is 7 lbs'

In [83]:
#result rturned by the LLM/Assistant
zarvis(next_prompt)

'Answer: A toy poodle weighs 7 lbs.'

In [84]:
zarvis.messages

[{'role': 'system',
  'content': 'You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE. You have to PAUSE and avoid making up things.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\naverage_dog_weight:\ne.g. average_dog_weight: Collie\nreturns average weight of a dog when given the breed\n\nExample session:\n\nQuestion: How much does a Bulldog weigh?\nThought: I should look the dogs weight using average_dog_weight\nAction: average_dog_weight: Bulldog\nPAUSE\n\nYou will be called again with this:\n\nObservation: A Bulldog weights 51 lbs\n\nYou then output:\n\nAnswer: A bulldog weights 51 lbs'},


###  2 step Query to find avg weight of 2 dogs




In [56]:
zarvis = Agent(prompt)

In [57]:
zarvis.messages

[{'role': 'system',
  'content': 'You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE. You have to PAUSE and avoid making up things.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\naverage_dog_weight:\ne.g. average_dog_weight: Collie\nreturns average weight of a dog when given the breed\n\nExample session:\n\nQuestion: How much does a Bulldog weigh?\nThought: I should look the dogs weight using average_dog_weight\nAction: average_dog_weight: Bulldog\nPAUSE\n\nYou will be called again with this:\n\nObservation: A Bulldog weights 51 lbs\n\nYou then output:\n\nAnswer: A bulldog weights 51 lbs'}]

In [58]:
question = """I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight"""
zarvis(question)

'Thought: I need to find the weight of a Border Collie and a Scottish Terrier, then add them together.\nAction: average_dog_weight: Border Collie\nPAUSE'

In [59]:
zarvis.messages

[{'role': 'system',
  'content': 'You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE. You have to PAUSE and avoid making up things.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\naverage_dog_weight:\ne.g. average_dog_weight: Collie\nreturns average weight of a dog when given the breed\n\nExample session:\n\nQuestion: How much does a Bulldog weigh?\nThought: I should look the dogs weight using average_dog_weight\nAction: average_dog_weight: Bulldog\nPAUSE\n\nYou will be called again with this:\n\nObservation: A Bulldog weights 51 lbs\n\nYou then output:\n\nAnswer: A bulldog weights 51 lbs'},


In [60]:
#executed by agent workflow
next_prompt = "Observation: {}".format(average_dog_weight("Border Collie"))
print(next_prompt)

Observation: a Border Collies average weight is 37 lbs


In [61]:
zarvis(next_prompt)

'Thought: Now I need to find the weight of a Scottish Terrier and add it to the weight of the Border Collie.\nAction: average_dog_weight: Scottish Terrier\nPAUSE'

In [62]:
#executed by agent workflow
next_prompt = "Observation: {}".format(average_dog_weight("Scottish Terrier"))
print(next_prompt)

Observation: Scottish Terriers average 20 lbs


In [63]:
zarvis(next_prompt)

'Thought: Now I need to add the weights of the Border Collie and the Scottish Terrier together.\nAction: calculate: 37 + 20\nPAUSE'

In [64]:
next_prompt = "Observation: {}".format(eval("37 + 20"))
print(next_prompt)

Observation: 57


In [65]:
zarvis(next_prompt)

'Answer: The combined weight of a Border Collie and a Scottish Terrier is 57 lbs.'

## Automating the process using a loop


In [66]:
# \w = letters/numbers/underscore
# +  = one or more characters

#used to match Action: average_dog_weight: Border Collie

action_re = re.compile('^Action: (\w+): (.*)$')   # python regular expression to selection action

<>:1: SyntaxWarning: invalid escape sequence '\w'
<>:1: SyntaxWarning: invalid escape sequence '\w'
/tmp/ipykernel_233/2389047760.py:1: SyntaxWarning: invalid escape sequence '\w'
  action_re = re.compile('^Action: (\w+): (.*)$')   # python regular expression to selection action


In [69]:
def query(question, max_turns=5):
    i = 0
    bot = Agent(prompt)
    next_prompt = question
    while i < max_turns:
        print("\n****Iteration: {}\n".format(i))
        i += 1
        result = bot(next_prompt)
        print(result)
        # we expect only 1 action
        actions = [
            action_re.match(a)
            for a in result.split('\n')
            if action_re.match(a)
        ]
        if actions:
            # There is an action to run
            action, action_input = actions[0].groups()
            if action not in known_actions:
                raise Exception("Unknown action: {}: {}".format(action, action_input))
            print(" -- running {} {}".format(action, action_input))
            observation = known_actions[action](action_input)
            print("Observation:", observation)
            next_prompt = "Observation: {}".format(observation)
        else:
            return

In [70]:
question = """I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight"""
query(question)


****Iteration: 0

Thought: I need to find the weight of a Border Collie and a Scottish Terrier, then add them together.
Action: average_dog_weight: Border Collie
PAUSE
 -- running average_dog_weight Border Collie
Observation: a Border Collies average weight is 37 lbs

****Iteration: 1

Thought: Now I need to find the weight of a Scottish Terrier and add it to the weight of the Border Collie.
Action: average_dog_weight: Scottish Terrier
PAUSE
 -- running average_dog_weight Scottish Terrier
Observation: Scottish Terriers average 20 lbs

****Iteration: 2

Thought: Now I need to add the weights of the Border Collie and the Scottish Terrier together.
Action: calculate: 37 + 20
PAUSE
 -- running calculate 37 + 20
Observation: 57

****Iteration: 3

Answer: The combined weight of a Border Collie and a Scottish Terrier is 57 lbs.
